# 손 키포인트 모델 재학습 (양손 chirality 편향 교정) — 무료 Colab

현재 모델은 미완성(~50%)이라 **한쪽 손만** 잘 인식합니다. 손과 그 거울상은
좌/우(chirality)라서, **좌우반전 증강(`fliplr`)을 켜고 끝까지 학습**하면 양손이
모두 잡힙니다. 이미지는 직접 모을 필요 없이 Ultralytics `hand-keypoints`
데이터셋(~26k장, 21키포인트, 양손)이 자동 다운로드됩니다.

**무료 Colab 주의:** 런타임 → 런타임 유형 변경 → **GPU(T4)** 선택. 장시간
학습 중 끊길 수 있어 가중치를 **Google Drive**에 저장하고, 끊기면 RESUME 셀로
이어서 학습합니다.

끝나면 `best.onnx`(opset 12)가 만들어지고, 이를 `Colab_hand_to_rknn.ipynb`로
변환해 `models/hand_pose_640.rknn`을 교체하면 됩니다.

In [ ]:
# 1) GPU 확인 (없으면: 런타임 → 런타임 유형 변경 → GPU)
!nvidia-smi -L

In [ ]:
# 2) Google Drive 마운트 (끊겨도 가중치 보존)
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/ai_curtain_train'   # 학습 결과 저장 위치
NAME = 'hand_pose'
print('weights ->', f'{PROJECT}/{NAME}/weights/best.pt')

In [ ]:
# 3) Ultralytics 설치
!pip -q install ultralytics
import ultralytics; ultralytics.checks()

## 4) 학습 (최초 1회)
`fliplr=0.5`가 핵심 — 양손(좌/우 chirality)을 균등 학습합니다. 무료 T4 기준
수 시간 걸릴 수 있으니 `patience=20`으로 수렴 시 조기 종료합니다.
**중간에 끊기면 이 셀 대신 아래 RESUME 셀을 실행하세요.**

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n-pose.pt')          # 작고 빠른 nano (무료 Colab 적합)
model.train(
    data='hand-keypoints.yaml',          # 자동 다운로드 (~26k장, 21kp, 양손)
    epochs=100,
    imgsz=640,                           # 보드 프로파일(hand_near)과 동일
    batch=16,
    fliplr=0.5,                          # <-- 좌우반전 증강 = 양손 편향 교정
    patience=20,                         # 수렴 시 조기 종료
    project=PROJECT, name=NAME,
)
print('best:', f'{PROJECT}/{NAME}/weights/best.pt')

## 4-R) RESUME — 끊겼을 때만 실행
런타임 재연결 후 2)·3) 셀(마운트·설치)을 다시 돌린 다음 이 셀을 실행하면
Drive의 `last.pt`에서 이어서 학습합니다. (4 셀은 다시 돌리지 마세요)

In [ ]:
from ultralytics import YOLO
model = YOLO(f'{PROJECT}/{NAME}/weights/last.pt')
model.train(resume=True)

## 5) ONNX 내보내기 (opset 12)
보드 `postprocess.py`가 이 표준 레이아웃을 자동 인식합니다.

In [ ]:
from ultralytics import YOLO
best = f'{PROJECT}/{NAME}/weights/best.pt'
onnx = YOLO(best).export(format='onnx', opset=12, imgsz=640)
print('ONNX:', onnx)   # 같은 폴더에 best.onnx 생성 (Drive에 보존됨)

## 6) 다음 단계 — RKNN 변환
위 `best.onnx`를 **`Colab_hand_to_rknn.ipynb`**(또는 `convert_rknn.py`)로 변환:

```
python convert_rknn.py best.onnx --target rk3576 --out hand_pose_640.rknn
```

생성된 `hand_pose_640.rknn`을 보드의 `models/hand_pose_640.rknn`에 덮어쓰고
앱을 재시작하면 양손이 모두 인식됩니다. (변환은 의존성 버전이 까다로워 학습과
**별도 런타임**에서 도는 기존 변환 노트북을 그대로 쓰세요.)